# A15 — GAN-Based Synthetic Handwritten Digit Generation
**Course:** 102903/CO701B Deep Learning — Rajagiri School of Engineering & Technology  
**Team:** Amrith Saras · Annabel Marianne Victor · Diya Jothish  
**Module:** 3 — Generative Adversarial Networks

---

## Problem Statement
Train a Deep Convolutional GAN (DCGAN) on the MNIST dataset to generate realistic synthetic handwritten digit images. Evaluate generation quality visually and through training loss dynamics.

## Objectives
1. Implement a DCGAN with a convolutional Generator and Discriminator.
2. Train the model on MNIST (70,000 greyscale 28×28 digit images).
3. Generate novel synthetic digit images and analyse their quality.
4. Track and visualise Generator and Discriminator loss across epochs.
5. Save the trained model for future inference.

## 1. Imports and Environment Setup

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'Torchvision version: {torchvision.__version__}')

os.makedirs('outputs/samples', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)

## 2. Hyperparameters

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
LATENT_DIM   = 100      # size of the noise vector fed to Generator
IMAGE_SIZE   = 28       # MNIST image dimensions
CHANNELS     = 1        # greyscale
BATCH_SIZE   = 128
NUM_EPOCHS   = 50
LR           = 0.0002   # Adam learning rate (DCGAN paper recommendation)
BETA1        = 0.5      # Adam beta1 (DCGAN paper recommendation)
BETA2        = 0.999
FEATURES_G   = 64       # base feature map size for Generator
FEATURES_D   = 64       # base feature map size for Discriminator
SAMPLE_EVERY = 5        # save sample grid every N epochs

print('Hyperparameters set.')

## 3. Dataset — MNIST

MNIST contains **70,000** greyscale 28×28 images of handwritten digits (0–9).  
We normalise pixel values from [0,1] → [−1,1] to match the Generator's Tanh output.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])   # maps [0,1] → [−1,1]
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

print(f'Training samples : {len(train_dataset):,}')
print(f'Batches per epoch: {len(train_loader)}')

# Visualise a batch of real images
real_batch, _ = next(iter(train_loader))
grid = make_grid(real_batch[:64], nrow=8, normalize=True)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.title('Sample Real MNIST Images', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('outputs/samples/real_samples.png', dpi=150)
plt.show()

## 4. Model Architecture

### 4.1 Background — How a GAN Works

A **Generative Adversarial Network** (Goodfellow et al., 2014) consists of two networks trained simultaneously in a minimax game:

| Network | Role | Input | Output |
|---------|------|-------|--------|
| **Generator G** | Creates fake images | Random noise vector **z** | Fake image |
| **Discriminator D** | Distinguishes real vs fake | Image | Probability ∈ (0,1) |

**Minimax objective:**
$$\min_G \max_D \; \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))]$$

- **D** maximises: correctly labelling real images as real and fakes as fake.
- **G** minimises: making D label its outputs as real.

### 4.2 DCGAN Improvements (Radford et al., 2015)
- Replace pooling with **strided convolutions** (D) and **transposed convolutions** (G).
- **Batch Normalisation** in both G and D (except G output and D input).
- **LeakyReLU** in D; **ReLU** in G hidden layers; **Tanh** at G output.

### Architecture Diagram

```
GENERATOR
z (100) → Linear → Reshape(256,7,7)
         → ConvTranspose(128, k=4, s=2, p=1) → 128×14×14  [BN+ReLU]
         → ConvTranspose( 64, k=4, s=2, p=1) →  64×28×28  [BN+ReLU]
         → Conv(1, k=3, s=1, p=1)            →   1×28×28  [Tanh]

DISCRIMINATOR
1×28×28 → Conv(64,  k=4, s=2, p=1) →  64×14×14  [LeakyReLU(0.2)]
         → Conv(128, k=4, s=2, p=1) → 128× 7× 7  [BN+LeakyReLU(0.2)]
         → Conv(256, k=4, s=2, p=1) → 256× 3× 3  [BN+LeakyReLU(0.2)]
         → Flatten → Linear(1)                   [Sigmoid]
```

In [ ]:
class Generator(nn.Module):
    """DCGAN Generator: noise vector → 1×28×28 fake image."""

    def __init__(self, latent_dim, features_g, channels):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(latent_dim, features_g * 4 * 7 * 7, bias=False),
            nn.BatchNorm1d(features_g * 4 * 7 * 7),
            nn.ReLU(inplace=True)
        )
        self.conv_blocks = nn.Sequential(
            # 256×7×7 → 128×14×14
            nn.ConvTranspose2d(features_g * 4, features_g * 2,
                               kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(inplace=True),
            # 128×14×14 → 64×28×28
            nn.ConvTranspose2d(features_g * 2, features_g,
                               kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(inplace=True),
            # 64×28×28 → 1×28×28
            nn.Conv2d(features_g, channels,
                      kernel_size=3, stride=1, padding=1, bias=False),
            nn.Tanh()   # output in [−1, 1]
        )

    def forward(self, z):
        x = self.project(z)
        x = x.view(x.size(0), -1, 7, 7)   # reshape to spatial feature map
        return self.conv_blocks(x)


class Discriminator(nn.Module):
    """DCGAN Discriminator: 1×28×28 image → real/fake probability."""

    def __init__(self, channels, features_d):
        super().__init__()
        self.model = nn.Sequential(
            # 1×28×28 → 64×14×14  (no BN on first layer — DCGAN convention)
            nn.Conv2d(channels, features_d,
                      kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 64×14×14 → 128×7×7
            nn.Conv2d(features_d, features_d * 2,
                      kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # 128×7×7 → 256×3×3
            nn.Conv2d(features_d * 2, features_d * 4,
                      kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # 256×3×3 → 1 (scalar)
            nn.Flatten(),
            nn.Linear(features_d * 4 * 3 * 3, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


def init_weights(m):
    """DCGAN weight initialisation: N(0, 0.02) for Conv/Linear layers."""
    classname = m.__class__.__name__
    if 'Conv' in classname or 'Linear' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# Instantiate
G = Generator(LATENT_DIM, FEATURES_G, CHANNELS).to(DEVICE)
D = Discriminator(CHANNELS, FEATURES_D).to(DEVICE)

G.apply(init_weights)
D.apply(init_weights)

print('Generator architecture:')
print(G)
total_g = sum(p.numel() for p in G.parameters())
print(f'\nGenerator parameters : {total_g:,}')

print('\nDiscriminator architecture:')
print(D)
total_d = sum(p.numel() for p in D.parameters())
print(f'\nDiscriminator parameters: {total_d:,}')

## 5. Loss Function and Optimisers

**Loss:** Binary Cross-Entropy (BCE)
$$\mathcal{L}_{BCE} = -[y \log(\hat{y}) + (1-y)\log(1-\hat{y})]$$

**Training procedure per batch:**
1. **Update D:** maximise `log D(x) + log(1 − D(G(z)))`
   - Real images → D should output 1 (real)
   - Fake images → D should output 0 (fake)
2. **Update G:** maximise `log D(G(z))` (equivalently, minimise `log(1 − D(G(z)))`)
   - Feed fake images to D; G wants D to output 1 (real)

In [ ]:
criterion = nn.BCELoss()

opt_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2))

# Fixed noise for consistent visualisation across epochs
fixed_noise = torch.randn(64, LATENT_DIM, device=DEVICE)

print('Loss function and optimisers ready.')

## 6. Training Loop

In [ ]:
G_losses, D_losses = [], []
D_real_acc, D_fake_acc = [], []

REAL_LABEL = 1.0
FAKE_LABEL = 0.0

print(f'Starting training for {NUM_EPOCHS} epochs on {DEVICE}...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_g_loss, epoch_d_loss = 0.0, 0.0
    epoch_d_real, epoch_d_fake = 0.0, 0.0

    for batch_idx, (real_imgs, _) in enumerate(train_loader):
        real_imgs = real_imgs.to(DEVICE)
        bsz = real_imgs.size(0)

        # ── Train Discriminator ───────────────────────────────────────────────
        D.zero_grad()

        # Real images — label = 1
        real_labels = torch.full((bsz, 1), REAL_LABEL, device=DEVICE)
        out_real = D(real_imgs)
        loss_d_real = criterion(out_real, real_labels)

        # Fake images — label = 0
        z = torch.randn(bsz, LATENT_DIM, device=DEVICE)
        fake_imgs = G(z)
        fake_labels = torch.full((bsz, 1), FAKE_LABEL, device=DEVICE)
        out_fake = D(fake_imgs.detach())   # detach: don't backprop into G yet
        loss_d_fake = criterion(out_fake, fake_labels)

        loss_D = loss_d_real + loss_d_fake
        loss_D.backward()
        opt_D.step()

        # ── Train Generator ───────────────────────────────────────────────────
        G.zero_grad()

        # G wants D to classify fakes as real → use real_labels
        out_fake_for_g = D(fake_imgs)
        loss_G = criterion(out_fake_for_g, real_labels)
        loss_G.backward()
        opt_G.step()

        # Accumulate for logging
        epoch_g_loss += loss_G.item()
        epoch_d_loss += loss_D.item()
        epoch_d_real += out_real.mean().item()
        epoch_d_fake += out_fake.mean().item()

    # Per-epoch averages
    n = len(train_loader)
    G_losses.append(epoch_g_loss / n)
    D_losses.append(epoch_d_loss / n)
    D_real_acc.append(epoch_d_real / n)
    D_fake_acc.append(epoch_d_fake / n)

    print(f'Epoch [{epoch:>3}/{NUM_EPOCHS}] '
          f'Loss_D: {D_losses[-1]:.4f}  Loss_G: {G_losses[-1]:.4f}  '
          f'D(x): {D_real_acc[-1]:.3f}  D(G(z)): {D_fake_acc[-1]:.3f}')

    # Save sample grid every SAMPLE_EVERY epochs
    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        G.eval()
        with torch.no_grad():
            fake_samples = G(fixed_noise).cpu()
        G.train()
        grid = make_grid(fake_samples, nrow=8, normalize=True)
        plt.figure(figsize=(8, 8))
        plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
        plt.title(f'Generated Digits — Epoch {epoch}', fontsize=13)
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(f'outputs/samples/epoch_{epoch:03d}.png', dpi=150)
        plt.show()

print('\nTraining complete.')

## 7. Training Loss Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(epochs, G_losses, label='Generator Loss', color='royalblue', linewidth=2)
axes[0].plot(epochs, D_losses, label='Discriminator Loss', color='tomato', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Generator vs Discriminator Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# D outputs
axes[1].plot(epochs, D_real_acc, label='D(x) — real images', color='green', linewidth=2)
axes[1].plot(epochs, D_fake_acc, label='D(G(z)) — fake images', color='orange', linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Equilibrium = 0.5')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Discriminator Output')
axes[1].set_title('Discriminator Confidence Over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('DCGAN Training Dynamics on MNIST', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/samples/training_curves.png', dpi=150)
plt.show()

print(f'Final Generator Loss     : {G_losses[-1]:.4f}')
print(f'Final Discriminator Loss : {D_losses[-1]:.4f}')
print(f'Final D(x)               : {D_real_acc[-1]:.4f}  (ideal ≈ 0.5)')
print(f'Final D(G(z))            : {D_fake_acc[-1]:.4f}  (ideal ≈ 0.5)')

## 8. Final Generated Samples

In [ ]:
G.eval()
with torch.no_grad():
    # Generate 100 random digits
    random_noise = torch.randn(100, LATENT_DIM, device=DEVICE)
    generated = G(random_noise).cpu()

grid = make_grid(generated, nrow=10, normalize=True, padding=2)
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.title('100 GAN-Generated Handwritten Digits (Final Model)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('outputs/samples/final_generated_100.png', dpi=150)
plt.show()

## 9. Side-by-Side Comparison: Real vs Generated

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Real
real_batch, _ = next(iter(train_loader))
real_grid = make_grid(real_batch[:32], nrow=16, normalize=True)
axes[0].imshow(real_grid.permute(1, 2, 0).numpy(), cmap='gray')
axes[0].set_title('Real MNIST Images', fontsize=12)
axes[0].axis('off')

# Generated
with torch.no_grad():
    fake_batch = G(torch.randn(32, LATENT_DIM, device=DEVICE)).cpu()
fake_grid = make_grid(fake_batch, nrow=16, normalize=True)
axes[1].imshow(fake_grid.permute(1, 2, 0).numpy(), cmap='gray')
axes[1].set_title('GAN-Generated Images', fontsize=12)
axes[1].axis('off')

plt.suptitle('Real vs GAN-Generated MNIST Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/samples/real_vs_fake.png', dpi=150)
plt.show()

## 10. Latent Space Interpolation

Interpolating between two noise vectors **z₁** and **z₂** reveals that the Generator has learned a smooth, meaningful latent space.

In [ ]:
G.eval()
STEPS = 10

z1 = torch.randn(1, LATENT_DIM, device=DEVICE)
z2 = torch.randn(1, LATENT_DIM, device=DEVICE)

alphas = torch.linspace(0, 1, STEPS)
interp_imgs = []

with torch.no_grad():
    for alpha in alphas:
        z_interp = (1 - alpha) * z1 + alpha * z2
        img = G(z_interp).cpu()
        interp_imgs.append(img)

interp_grid = make_grid(torch.cat(interp_imgs), nrow=STEPS, normalize=True)
plt.figure(figsize=(15, 2))
plt.imshow(interp_grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.title('Latent Space Interpolation (z₁ → z₂)', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig('outputs/samples/latent_interpolation.png', dpi=150)
plt.show()
print('Smooth transitions indicate the Generator has learned a meaningful latent space.')

## 11. Save Trained Models

In [ ]:
torch.save(G.state_dict(), 'outputs/models/generator.pth')
torch.save(D.state_dict(), 'outputs/models/discriminator.pth')
print('Models saved to outputs/models/')

## 12. Load and Inference (standalone use)

In [ ]:
# Load saved Generator and generate new digits
G_loaded = Generator(LATENT_DIM, FEATURES_G, CHANNELS).to(DEVICE)
G_loaded.load_state_dict(torch.load('outputs/models/generator.pth', map_location=DEVICE))
G_loaded.eval()

with torch.no_grad():
    z_new = torch.randn(16, LATENT_DIM, device=DEVICE)
    new_digits = G_loaded(z_new).cpu()

grid = make_grid(new_digits, nrow=8, normalize=True)
plt.figure(figsize=(8, 3))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.title('Newly Generated Digits from Loaded Model', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()
print('Inference from saved model successful.')

## 13. Results Analysis

### Observations

| Metric | Value / Observation |
|--------|---------------------|
| Training epochs | 50 |
| Generator loss (final) | *(fill after run)* |
| Discriminator loss (final) | *(fill after run)* |
| D(x) final | *(fill after run)* |
| D(G(z)) final | *(fill after run)* |
| Visual quality | Legible digits visible from ~epoch 10 |

### Analysis
- **Early epochs (1–10):** Generated images are noisy and blurry. D easily distinguishes real from fake — D_loss is low, G_loss is high.
- **Mid training (10–30):** G begins producing recognisable digit-like shapes. Both losses stabilise.
- **Late training (30–50):** Generated digits become sharp and varied. D(x) and D(G(z)) both approach 0.5, indicating the training has reached Nash equilibrium where D can no longer reliably distinguish real from fake.
- **Latent interpolation:** Smooth transitions between generated digits confirm the Generator has learned a structured, continuous latent space.
- **Mode diversity:** The 100-sample grid shows diversity across digit classes — the model has not collapsed to a single mode.

### Limitations
- GANs can suffer from **mode collapse** (generating only a subset of digit classes).
- Training instability is common — loss curves may oscillate.
- No quantitative metric like FID (Fréchet Inception Distance) computed here due to compute constraints.

## Conclusion
A DCGAN was successfully trained on MNIST to generate synthetic handwritten digit images. The Generator learned to map 100-dimensional Gaussian noise to realistic 28×28 digit images over 50 epochs. Training dynamics showed expected GAN behaviour: competitive loss convergence and increasing visual fidelity. The latent space interpolation demonstrated smooth, continuous digit generation.

## Future Scope
- **Conditional GAN (cGAN):** Condition on digit class to control which digit is generated.
- **Wasserstein GAN (WGAN):** More stable training using Earth Mover distance.
- **FID score:** Quantitatively evaluate generation quality.
- **Progressive growing GAN:** Generate higher-resolution images.

## References
1. Goodfellow, I. et al. (2014). *Generative Adversarial Nets.* NeurIPS.
2. Radford, A., Metz, L., & Chintala, S. (2015). *Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks.* arXiv:1511.06434.
3. LeCun, Y. et al. (1998). *Gradient-Based Learning Applied to Document Recognition.* (MNIST dataset)
4. Salimans, T. et al. (2016). *Improved Techniques for Training GANs.* NeurIPS.